# Error Trend Plot

From Craig Lage's notebook collection

In [ ]:
from datetime import date

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.time import Time, TimeDelta

In [ ]:
# TODO: Preguntar por la nueva ubicación de los archivos
DATA_DIR = "/path/to/correct/directory"

def read_rubin_tv_json(day_obs, camera):
    """
    Reads a RubinTV JSON file for the given observation day and camera.

    Parameters:
    ----------
    day_obs : int
        Observation day in the format YYYYMMDD.
    camera : str
        Camera type: 'Wide', 'Narrow', or 'AuxTel'.

    Returns:
    -------
    pd.DataFrame
        Transposed DataFrame containing the JSON data.
    """
    # Convert day_obs to ISO format using astropy
    time_obj = getDayObsStartTime(day_obs)
    date_str = time_obj.strftime("%Y-%m-%d")

    # Define filename based on camera type
    filename_map = {
        "Wide": f"{DATA_DIR}/startracker-wide_{date_str}.json",
        "Narrow": f"{DATA_DIR}/startracker_{date_str}.json",
        "AuxTel": f"{DATA_DIR}/auxtel_{date_str}.json",
    }

    filename = filename_map.get(camera)
    if not filename:
        raise ValueError(f"Invalid camera type: {camera}")

    try:
        df = pd.read_json(filename).transpose()
        print(f"Loaded file: {filename}")
        return df
    except FileNotFoundError:
        print(f"Error: File not found - {filename}")
        return None


def read_old_star_tracker_files(day_obs, camera):
    """
    Reads an old-style Star Tracker summary file.

    Parameters:
    ----------
    day_obs : int
        Observation day in the format YYYYMMDD.
    camera : str
        Camera type.

    Returns:
    -------
    np.array
        Data array from the file.
    """
    filename = f"{DATA_DIR}/StarTracker_Summary_{camera}_{day_obs}.txt"

    try:
        data = np.loadtxt(filename, skiprows=1)
        print(f"Loaded file: {filename}")
        return data
    except FileNotFoundError:
        print(f"Error: File not found - {filename}")
        return None


In [ ]:
# List of observation dates in YYYYMMDD format
dates = [
    20221123, 20221124, 20221128, 20221207, 20230130, 
    20230220, 20230221, 20230222, 20230307, 20230308, 20230309
]

camera = "Narrow"
num = 102

# Variables to store results
x_axis = []
delta_azs = []
delta_els = []
counter = 1

for day_obs in dates:
    # Convert observation date to a time object
    time_obj = getDayObsStartTime(day_obs)

    if day_obs < 20230220:
        # Read data from old StarTracker files
        data = read_old_star_tracker_files(day_obs, camera)
        if data is None:
            continue  # Skip if the file does not exist

        for row in data:
            (
                seq_num, ra, dec, ra_solve, dec_solve, az, el, 
                az_solve, el_solve, delta_az, delta_el, 
                rot, rms_error
            ) = row
            
            x_axis.append(counter)
            delta_azs.append(abs(delta_az) * 3600.0)  # Convert to arcseconds
            delta_els.append(abs(delta_el) * 3600.0)

    else:
        # Read data from RubinTV JSON files
        df = read_rubin_tv_json(day_obs, camera)
        if df is None:
            continue  # Skip if the file does not exist
        
        # Remove rows with NaN values
        df = df.dropna()

        for seq_num, row in df.iterrows():
            delta_alt = row["Delta Alt Arcsec"]
            delta_az = row["Delta Az Arcsec"]

            if day_obs < 20230309 or (day_obs == 20230309 and seq_num < 1138):
                x_axis.append(counter)
            elif day_obs == 20230309 and seq_num > 1138:
                x_axis.append(counter + 1)

            delta_azs.append(abs(delta_az))
            delta_els.append(abs(delta_alt))

    counter += 1


In [ ]:
len(xaxis)

In [ ]:
# Extend dates list to include the last observation date
plot_dates = dates + [20230309]

# Define x-axis ticks
x_ticks = np.arange(1, len(plot_dates) + 1, 1)

# Create the plot for StarTracker Narrow Azimuth Error Trend
plt.figure(figsize=(8, 6))
plt.title("StarTracker Narrow Azimuth Error Trend")

# Scatter plot of azimuth errors
plt.scatter(x_axis, delta_azs, color="blue", alpha=0.7, label="Azimuth Error")

# Use logarithmic scale for the y-axis
plt.yscale("log")
plt.ylim(1.0, 1.0e5)

# Set x-axis labels with proper rotation
plt.xticks(x_ticks, plot_dates, rotation=-45)

# Label axes
plt.xlabel("Date")
plt.ylabel("Azimuth Error (arcseconds)")

# Show grid and legend for better readability
plt.grid(True, which="both", linestyle="--", linewidth=0.5)
plt.legend()
plt.tight_layout()

# Display the plot
plt.show()


In [ ]:
def  readRubinTV_json(date, camera):
    year = int(date/10000)
    month = int((date - 10000 * year)/100)
    day = int((date - 10000 * year - 100 * month))
    if camera == 'Wide':
        filename = f'/scratch/cslage/starTracker/startracker-wide_{year}-{month:02}-{day:02}.json'
    elif camera == 'Narrow':
        filename = f'/scratch/cslage/starTracker/startracker_{year}-{month:02}-{day:02}.json'
    elif camera == 'AuxTel':
        filename = f'/scratch/cslage/starTracker/auxtel_{year}-{month:02}-{day:02}.json'
    df = pd.read_json(filename)
    df = df.transpose()
    print(filename)
    return df

def readOldStarTrackerFiles(date, camera):
        filename = f"/scratch/cslage/starTracker/StarTracker_Summary_{camera}_{date}.txt"
        data = np.loadtxt(filename, skiprows=1)
        return data


In [ ]:
[camera, num] = ['Narrow', 102]
xaxis = []
deltaazs = []
deltaels = []
dates = [20221123, 20221124, 20221128, 20221207, 20230130, 20230220,  20230221, 20230222, 20230307, 20230308, 20230309]
counter = 1
for date in dates:
    year = int(date/10000)
    month = int((date - 10000 * year)/100)
    day = int((date - 10000 * year - 100 * month))
    if date < 20230220:
        data = readOldStarTrackerFiles(date, camera)
        for j in range(data.shape[0]):
            [seqNum,ra,dec,raSolve,decSolve,Az,El,azSolve,elSolve,deltaAz,deltaEl,\
             rot,rms_error] = data[j]
            xaxis.append(counter)
            deltaazs.append(abs(deltaAz) * 3600.0)
            deltaels.append(abs(deltaEl) * 3600.0)
    else:
        df = readRubinTV_json(date, camera)
        df = df[~df.isnull().any(axis=1)]
        for seqNum in df.index.values.tolist():
            row = df.loc[seqNum]
            deltaAlt = row['Delta Alt Arcsec']
            deltaAz = row['Delta Az Arcsec']
            if date < 20230309 or (date == 20230309 and seqNum < 1138):
                xaxis.append(counter)
                deltaazs.append(abs(deltaAz))
                deltaels.append(abs(deltaAlt))
            if date == 20230309 and seqNum > 1138:
                xaxis.append(counter + 1)
                deltaazs.append(abs(deltaAz))
                deltaels.append(abs(deltaAlt))
                
    counter += 1


In [ ]:
len(xaxis)

In [ ]:
plot_dates = dates + [20230309]
xticks = np.arange(1,13,1)
plt.title("StarTracker Narrow Azimuth Error Trend")
plt.scatter(xaxis, deltaazs)
plt.yscale('log')
plt.ylim(1.0, 1.0E5)
plt.xticks(xticks, plot_dates, rotation=-45)
plt.xlabel("Date")
plt.ylabel("Azimuth Error (arcseconds)")
